# Task 07：TRPO 与 PPO——学习与实验入口

## 本 Notebook 的学习边界

PPO 用旧策略与新策略的概率比率控制更新幅度：

$$
r_t(	heta)=rac{\pi_	heta(A_t\mid S_t)}{\pi_{	heta_{old}}(A_t\mid S_t)},
$$

$$
L^{CLIP}=\mathbb{E}_t\left[\min\left(r_tA_t,\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_tight)ight].
$$

先理解 actor/critic 与 rollout，再观察 clip 对正负 advantage 的不同作用，最后比较 PPO、one-step Actor-Critic 和 GAE/clip 超参数。


In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
from torch.distributions import Categorical
import numpy as np
from pathlib import Path

In [4]:
env = gym.make(
    "CartPole-v1"
)

state, _ = env.reset()

print(state)

print("state dimension:", env.observation_space.shape)

print("action space:", env.action_space)

[ 0.00679818 -0.04244882 -0.00018896  0.01251047]
state dimension: (4,)
action space: Discrete(2)


In [5]:
class SimpleActor(nn.Module):

    def __init__(
        self,
        state_dim,
        action_dim,
    ):
        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                state_dim,
                64
            ),

            nn.Tanh(),

            nn.Linear(
                64,
                action_dim
            )
        )


    def forward(self,state):

        logits = self.net(state)

        return Categorical(
            logits=logits
        )

state_dim = env.observation_space.shape[0]

action_dim = env.action_space.n


actor = SimpleActor(
    state_dim,
    action_dim
)

In [7]:
state_tensor = torch.tensor(
    state,
    dtype=torch.float32
)

state_tensor = state_tensor.unsqueeze(0)

dist = actor(
    state_tensor
)

dist.probs

tensor([[0.4913, 0.5087]], grad_fn=<SoftmaxBackward0>)

In [8]:
action = dist.sample()

action

for _ in range(10):

    action = dist.sample()

    print(
        action.item()
    )

0
1
0
0
0
0
0
0
1
0


In [9]:
action = dist.sample()

log_prob = dist.log_prob(
    action
)

log_prob

tensor([-0.6759], grad_fn=<SqueezeBackward1>)

## 实验结论与提交要求

运行完本 Notebook 后，不要只保留图。请在对应 `notes/` 中记录随机种子、环境版本、关键超参数、最终指标、曲线文件和一个失败现象。结论必须区分“代码运行成功”和“算法表现更好”：前者由自检确认，后者需要多 seed 或控制变量实验支持。

提交前从仓库根目录运行：

```bash
python eval/run.py
```
